# 避難所候補算出支援ツール（sheltermatch）

要支援者一覧（緯度・経度入り）と避難所一覧の座標から、要支援者ごとに **距離が近い避難所候補（既定で上位3件）** を
算出し、必要であればハザード区域との位置関係も確認して、職員の最終判断のための資料（CSV）を作成するNotebookです。

**このツールが行うこと**
- 直線距離（`geopy.distance.geodesic`）による避難所候補の算出（候補提示。自動割当ではありません）
- 任意機能として、要支援者・避難所・両者を結ぶ直線とハザード区域（GeoJSON。国・県等の公式配布ZIPも
  展開せずそのままアップロード可能）との位置関係の確認

**このツールが行わないこと（重要）**
- 住所→座標変換（要支援者一覧CSVには緯度・経度をあらかじめ入力してください。`address` 列を含めても
  構いませんが、座標の補完には使用しません）
- 避難先・避難経路の自動決定、道路経路・通行可能性の計算（直線距離であり道路距離ではありません。
  直線とハザード区域の交差判定も、道路上の避難経路判定ではありません）
- ハザード判定結果による候補避難所の自動除外・自動順位変更、独自の危険度スコアリング

距離順位とハザード判定は別々の情報として出力します。どの避難所を選ぶかは、出力結果を確認した **職員が判断** してください。

**ハザード判定結果の読み方**
- `True` = ハザード区域内（境界上含む）または交差あり
- `False` = 有効な座標で判定した結果、ハザード区域外
- 空欄（NaN） = 座標が無い・不正、または候補自体が無いなどの理由で **判定できなかった**（区域外という意味ではありません）

**個人情報の取り扱い**
- 実際の要支援者データ・ハザードデータはこのリポジトリにコミットしないでください（サンプルを追加する場合は完全な架空データを使用してください）。
- 出力CSVには住所・座標等の個人情報が含まれ得ます。取り扱いに注意してください。

## 使い方

1. 下の「利用者設定」を確認する
2. 「ランタイム → すべてのセルを実行」
3. 要支援者CSV（`latitude` / `longitude` 列が必須です。値は行ごとに欠損・不正でも構いません）をアップロードする
4. 必要な場合のみハザードデータ（GeoJSONまたは国・県等の公式配布ZIP）をアップロードする
5. BODIK Data APIからの避難所取得に失敗した場合のみ、避難所CSVをアップロードする
6. 結果CSV（`assigned_shelters.csv`）を保存する

コードを読み込まなくても、この説明と各セルのprint出力だけで操作できます。


In [ ]:
# ===== 利用者設定 =====
# 通常変更が必要な項目はこれだけです。値を確認・変更してから実行してください。

# 要支援者・避難所とハザード区域（GeoJSON）との位置関係を確認する場合は True にしてください。
ENABLE_HAZARD_CHECK = False

# 要支援者ごとに算出する避難所候補の件数（避難所がこの件数未満の場合は存在する件数まで出力）
TOP_N = 3

# 避難所一覧の取得方法。"api"=BODIK Data APIから取得 / "csv"=CSVファイルをアップロード
# "api"で取得に失敗した場合は、自動的にCSVアップロードへ切り替わります。
SHELTER_SOURCE = "api"

if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError(f"TOP_N は1以上の整数を指定してください。現在の値: {TOP_N!r}")

if SHELTER_SOURCE not in ("api", "csv"):
    raise ValueError(f"SHELTER_SOURCE は 'api' または 'csv' を指定してください。現在の値: {SHELTER_SOURCE!r}")

print("利用者設定を読み込みました。")
print(f"  ENABLE_HAZARD_CHECK = {ENABLE_HAZARD_CHECK}")
print(f"  TOP_N               = {TOP_N}")
print(f"  SHELTER_SOURCE      = '{SHELTER_SOURCE}'")


In [ ]:
# ===== 実行環境準備 =====
# Google Colabに標準で入っていないライブラリをインストールします（初回のみ数十秒かかることがあります）。
%pip install -q geopy geopandas shapely

import io
import tempfile
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from geopy.distance import geodesic

import geopandas as gpd
from shapely.geometry import Point, LineString

from google.colab import files

print("ライブラリの読み込みが完了しました。")


In [ ]:
# ===== 要支援者CSV読込・入力チェック =====
# 要支援者一覧CSVを選択してください（ファイル名は自由です）。latitude / longitude 列が必須です
# （address や resident_id 等の追加列があっても、そのまま結果CSVに保持されます）。
# 文字コードは UTF-8 (BOM付き) → CP932 → UTF-8 の順に自動判定します。
# 座標が欠損・不正な行があっても削除せず、後続の距離計算のみ対象外とします
# （match_status 列で no_coordinates / invalid_coordinates として確認できます）。

print("【要支援者一覧CSV】を選択してください。")
uploaded_residents = files.upload()

if len(uploaded_residents) != 1:
    raise RuntimeError("要支援者一覧CSVは1つだけ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents_bytes = uploaded_residents[residents_filename]
print(f"'{residents_filename}' を要支援者一覧として受け取りました。")


def read_csv_auto(file_bytes, label):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試み、成功したDataFrameを返す。"""
    encodings = [
        ("utf-8-sig", "UTF-8 (BOM付き)"),
        ("cp932", "CP932 (Shift-JIS系)"),
        ("utf-8", "UTF-8"),
    ]
    last_error = None
    for encoding, encoding_label in encodings:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=encoding)
            print(f"[{label}] {encoding_label} として読み込みました。（{len(df)}行）")
            return df
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(
        f"[{label}] 文字コードを判定できませんでした。"
        "UTF-8(BOM付き)・CP932・UTF-8のいずれでも読み込めません。"
        "Excel等での保存時の文字コードを確認してください。"
        f" 詳細: {last_error}"
    )


def ensure_coordinate_columns(df, label):
    """latitude/longitude列が両方存在することを確認する（値の欠損・不正は許容し、ここでは
    行を削除しない。列自体が無い場合のみ列不足として停止する）。"""
    missing = [c for c in ("latitude", "longitude") if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}")
    return df


def is_valid_coordinate(lat, lon):
    """緯度・経度が数値として有効な範囲かどうかを返す（欠損・範囲外はFalse）。
    距離計算・ハザード判定など、1点ずつ座標を扱う関数から共通で利用する。"""
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def parse_and_validate_coordinates(df, label):
    """latitude/longitude列を数値化し、有効な座標かどうかの真偽値Seriesを返す。"""
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")
    valid = lat.notna() & lon.notna() & lat.between(-90, 90) & lon.between(-180, 180)
    invalid_count = int((~valid).sum())
    if invalid_count:
        print(f"[{label}] 座標が欠損・不正な行が {invalid_count}件あります（全{len(df)}行中）。")
    return lat, lon, valid


residents_raw = read_csv_auto(residents_bytes, "要支援者一覧")
residents_raw = ensure_coordinate_columns(residents_raw, "要支援者一覧")

residents = residents_raw.copy()
residents["latitude"], residents["longitude"], residents_coord_valid = parse_and_validate_coordinates(
    residents, "要支援者一覧"
)

display(residents_raw.head())


In [ ]:
# ===== 避難所取得・正規化・入力チェック =====
# 避難所一覧は上の「利用者設定」の SHELTER_SOURCE に従って取得します。
# - "api"（既定）: BODIK Data API（CKANの datastore_search）から自治体標準ODSの避難所データを
#   直接取得します（公開データの読み取りのみのためAPIキーは不要）。取得に失敗した場合
#   （通信エラー・レスポンス異常・0件など）は、Notebookを停止せずCSVアップロードへ自動的に切り替えます。
# - "csv": 避難所一覧CSVをブラウザから選択してアップロードします。
#
# 避難所一覧CSVが自治体標準オープンデータセット(ODS)形式（例:
# https://data.bodik.jp/dataset/472107_evacuation_space ）の場合、日本語列名
# （名称→name、緯度→latitude、経度→longitude）を自動的に内部標準列名へ変換します。
# 従来の name/latitude/longitude 形式のCSVもそのまま利用できます。
# 「災害種別_」で始まる列がある場合は、値の 1(対応済み)/2(2階以上であれば対応済み)/空欄(未対応) の
# 区別を保ったまま、避難所候補ごとにCSVへ参考情報として出力します（距離順位や候補の自動除外には
# 使用しません。これは指定緊急避難場所としての対応種別情報であり、後述のGeoJSONによるハザード判定
# とは別の情報です）。
# 座標が不正な避難所の行は距離計算の対象から除外します。

BODIK_BASE_URL = "https://data.bodik.jp"
BODIK_RESOURCE_ID = "3132a0a4-f522-4b2d-bf18-f106d8b3a5ae"  # 糸満市 指定緊急避難場所データセット


def fetch_shelters_from_bodik(base_url, resource_id, page_size=1000):
    """BODIKのCKAN Data API(datastore_search)から避難所データを全件取得し、DataFrameで返す。
    total/offsetでページングして全件取得する。取得できない場合は例外を発生させ、
    呼び出し側でCSVアップロードへフォールバックする。"""
    endpoint = f"{base_url}/api/action/datastore_search"
    records = []
    offset = 0
    total = None

    while True:
        response = requests.get(
            endpoint,
            params={"resource_id": resource_id, "limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        if not payload.get("success"):
            raise RuntimeError("CKAN APIレスポンスが success=false を返しました。")

        result = payload.get("result")
        if result is None or "records" not in result:
            raise RuntimeError("CKAN APIレスポンスに result.records が含まれていません。")

        page_records = result["records"]
        records.extend(page_records)

        if total is None:
            total = result.get("total", len(page_records))

        offset += len(page_records)
        if len(page_records) == 0 or offset >= total:
            break

    if len(records) == 0:
        raise RuntimeError("BODIK APIの取得結果が0件でした。")

    return pd.DataFrame(records)


shelters_raw = None
shelters_bytes = None

if SHELTER_SOURCE == "api":
    try:
        shelters_raw = fetch_shelters_from_bodik(BODIK_BASE_URL, BODIK_RESOURCE_ID)
        print(f"BODIK APIから避難所一覧を{len(shelters_raw)}件取得しました。")
    except Exception as error:
        print("BODIK APIから避難所一覧を取得できませんでした。")
        print("CSVファイルから読み込みます。")
        print(f"（詳細: {error}）")

if shelters_raw is None:
    print("【避難所一覧CSV】を選択してください。")
    uploaded_shelters = files.upload()

    if len(uploaded_shelters) != 1:
        raise RuntimeError("避難所一覧CSVは1つだけ選択してください。")

    shelters_filename = list(uploaded_shelters.keys())[0]
    shelters_bytes = uploaded_shelters[shelters_filename]
    print(f"'{shelters_filename}' を避難所一覧として受け取りました。")

if shelters_bytes is not None:
    shelters_raw = read_csv_auto(shelters_bytes, "避難所一覧")

# 避難所一覧の列名アイリアス（自治体標準ODS等の日本語列名 → 内部標準列名）
SHELTER_COLUMN_ALIASES = {
    "name": ["名称"],
    "latitude": ["緯度"],
    "longitude": ["経度"],
}

# 避難所の災害種別列の接頭辞（この接頭辞で始まる列を動的にすべて認識する）
DISASTER_TYPE_COLUMN_PREFIX = "災害種別_"

# 自治体標準ODSの災害種別列の値と対応区分の対応表（BODIKの指定緊急避難場所データセット仕様）
# 1=対応済み、2=2階以上であれば対応済み、空欄=未対応。これ以外の値は独自に「対応」と推測しない。
DISASTER_SUPPORT_LABELS = {
    "1": "対応済み",
    "1.0": "対応済み",
    "2": "2階以上であれば対応済み",
    "2.0": "2階以上であれば対応済み",
}


def normalize_shelter_columns(df, label):
    """自治体標準ODS等で使われる日本語列名を、内部標準列名(name/latitude/longitude)へ変換する。
    既に標準列名がある場合はそちらを優先し、変換しない。使用した列名の対応表も返す。"""
    df = df.copy()
    used_columns = {}
    for standard_col, aliases in SHELTER_COLUMN_ALIASES.items():
        if standard_col in df.columns:
            used_columns[standard_col] = standard_col
            continue
        for alias in aliases:
            if alias in df.columns:
                df = df.rename(columns={alias: standard_col})
                used_columns[standard_col] = alias
                break

    renamed = {std: orig for std, orig in used_columns.items() if orig != std}
    if renamed:
        mapping_text = ", ".join(f"{orig}→{std}" for std, orig in renamed.items())
        print(f"[{label}] 自治体標準ODS等の日本語列名を自動変換しました: {mapping_text}")

    return df, used_columns


def detect_disaster_type_columns(df):
    """列名が '災害種別_' で始まる列を、対応する災害種別情報として動的に検出する。
    特定の災害種別名の一覧に固定せず、実データに存在する列をそのまま採用する。"""
    return [c for c in df.columns if c.startswith(DISASTER_TYPE_COLUMN_PREFIX)]


def classify_disaster_support(value):
    """自治体標準ODSの災害種別列の値を解釈し、対応区分のラベルを返す（未対応ならNone）。
    '1'=対応済み、'2'=2階以上であれば対応済み、空欄=未対応という標準仕様に従う。
    それ以外の想定外の値は独自に「対応している」と推測せず、値をそのまま保持して要確認として返す。"""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text == "":
        return None
    if text in DISASTER_SUPPORT_LABELS:
        return DISASTER_SUPPORT_LABELS[text]
    return f"{text}(要確認)"


def build_disaster_support(df, disaster_columns):
    """行ごとに対応している災害種別とその対応区分を ';' 区切りでまとめたSeriesを返す。
    例: '洪水:対応済み;高潮:2階以上であれば対応済み'。未対応(空欄)の種別は含めない。
    災害種別列が無ければ全行NaN。"""
    if not disaster_columns:
        return pd.Series([np.nan] * len(df), index=df.index, dtype="object")

    prefix_len = len(DISASTER_TYPE_COLUMN_PREFIX)

    def row_support(row):
        parts = []
        for col in disaster_columns:
            label = classify_disaster_support(row[col])
            if label is not None:
                parts.append(f"{col[prefix_len:]}:{label}")
        return ";".join(parts)

    return df.apply(row_support, axis=1)


# 避難所一覧は座標列の存否を確認する前に、自治体標準ODS等の日本語列名を内部標準列名へ変換する
shelters_raw, shelter_used_columns = normalize_shelter_columns(shelters_raw, "避難所一覧")
shelter_disaster_columns = detect_disaster_type_columns(shelters_raw)
shelters_raw["_disaster_support"] = build_disaster_support(shelters_raw, shelter_disaster_columns)
HAS_DISASTER_TYPE_COLUMNS = len(shelter_disaster_columns) > 0

shelters_raw = ensure_coordinate_columns(shelters_raw, "避難所一覧")
if "name" not in shelters_raw.columns:
    raise ValueError("[避難所一覧] 名称列が見つかりません: name または 名称 の列が必要です。")

shelters = shelters_raw.copy()
shelters["latitude"], shelters["longitude"], shelters_coord_valid = parse_and_validate_coordinates(
    shelters, "避難所一覧"
)

# 座標が不正な避難所の行は距離計算の対象から除外する（有効な避難所が0件の場合は処理を停止する）
invalid_shelter_count = int((~shelters_coord_valid).sum())
shelters_valid = shelters.loc[shelters_coord_valid].reset_index(drop=True)

if len(shelters_valid) == 0:
    raise RuntimeError(
        "有効な座標を持つ避難所が0件です。避難所一覧CSVの latitude / longitude 列を確認してください。"
    )

print(
    f"[避難所一覧] 避難所件数: {len(shelters_raw)}件 / "
    f"名称列: '{shelter_used_columns.get('name', 'name')}' / "
    f"緯度列: '{shelter_used_columns.get('latitude', 'latitude')}' / "
    f"経度列: '{shelter_used_columns.get('longitude', 'longitude')}' / "
    f"認識した災害種別列数: {len(shelter_disaster_columns)}件"
)
print(f"距離計算に使用する有効な避難所: {len(shelters_valid)}件（座標不正のため{invalid_shelter_count}件を除外）")

display(shelters_raw.head())


## ハザードデータ読込（任意）

`ENABLE_HAZARD_CHECK = True`（上の「利用者設定」）の場合のみ、ハザード区域のデータをアップロードします。
1ファイルにつき **`.geojson` 単体**、または **国・県等の公式配布ZIP（`.zip`）** のいずれかを選択できます。
ZIPの場合、内部のディレクトリ構造やファイル数に関わらず、配下の `.geojson` を自動的に再帰探索してすべて
統合します（利用者が展開・整理する必要はありません）。洪水・土砂災害・津波・高潮等、複数のハザード種別を
まとめて選択でき、アップロードした **ファイル（GeoJSONまたはZIP）ごとに1回だけ** 基本となるハザード種別名
を入力します（ZIP内の個々のGeoJSONごとには入力しません）。

ZIPの展開先直下にサブフォルダがある場合（国土数値情報等で「計画規模」「想定最大規模」のようにカテゴリ別に
GeoJSONが分かれている場合）は、そのサブフォルダ名を **カテゴリ** として扱い、`基本種別名:カテゴリ名` の形で
`hazard_type` に保持します（例: 基本種別名「洪水」・サブフォルダ「42_家屋倒壊等氾濫想定区域_河岸侵食」→
`洪水:42_家屋倒壊等氾濫想定区域_河岸侵食`）。サブフォルダを持たないGeoJSON・ZIPは、入力した基本種別名が
そのまま `hazard_type` になります。フォルダ名の意味を独自解釈せずそのままカテゴリ名として使うため、特定の
配布元のファイル名・ディレクトリ名には依存しません。

座標系はWGS84（EPSG:4326）に統一され、Polygon/MultiPolygon以外の空・不正なジオメトリは除外されます。
`ENABLE_HAZARD_CHECK = False`（既定）の場合はこのセルはスキップされ、ハザードデータなしで距離候補算出のみ
が実行されます。

**注意**: `ENABLE_HAZARD_CHECK = True` にした場合、有効なハザード区域ポリゴンが1件も読み込めなかったときは
「ハザードなし」とみなさず、ここで処理を停止します（判定していないことと、ハザード区域でないことを区別するためです）。


In [ ]:
def safe_extract_zip(zip_bytes, extract_dir):
    """ZIPをextract_dirへ安全に展開する。絶対パスや'..'を含むなど、展開先ディレクトリの外に出る
    エントリが1件でもあれば、展開を一切行わずに例外を送出する（パストラバーサル対策。
    全エントリを先に検査してから展開する）。"""
    extract_dir_abs = Path(extract_dir).resolve()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        for member in zip_file.infolist():
            member_path = (extract_dir_abs / member.filename).resolve()
            if member_path != extract_dir_abs and extract_dir_abs not in member_path.parents:
                raise RuntimeError(
                    "ZIP内に不正なパスが含まれているため展開を中止しました"
                    f"（パストラバーサルの可能性）: {member.filename}"
                )
        zip_file.extractall(extract_dir_abs)


def load_hazard_geojson(source, hazard_type):
    """1件のGeoJSON（パスまたはファイルオブジェクト）を読み込み、hazard_type/geometryの2列に
    正規化し、WGS84(EPSG:4326)へ統一する。Polygon/MultiPolygon以外、または空・不正なジオメトリは
    除外する。"""
    gdf = gpd.read_file(source)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    elif gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    gdf = gdf[
        gdf.geometry.notna()
        & gdf.geometry.is_valid
        & gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    ]

    return gpd.GeoDataFrame(
        {"hazard_type": hazard_type, "geometry": gdf.geometry.values}, crs="EPSG:4326"
    )


def hazard_type_for_path(base_hazard_type, geojson_path, extract_dir):
    """ZIP展開先直下のサブフォルダ名を『カテゴリ』として保持したhazard_typeを返す。
    国土数値情報等では洪水の中でも「計画規模」「想定最大規模」等がサブフォルダで分かれて配布される
    ため、ZIP全体を1つのhazard_typeに潰すと区別が失われる。フォルダ名の意味は独自解釈せずそのまま
    カテゴリ名として使うため、特定の配布元のディレクトリ名には依存しない。展開先直下に置かれた
    ファイル（サブフォルダなし）はカテゴリなし（基本種別名のみ）として扱う。"""
    rel_parts = geojson_path.relative_to(extract_dir).parts
    if len(rel_parts) <= 1:
        return base_hazard_type
    category = rel_parts[0]
    return f"{base_hazard_type}:{category}"


def load_hazard_upload(filename, file_bytes, hazard_type):
    """1つのアップロード（.geojson または国・県等の公式配布ZIP）から、GeoDataFrameを組み立てる。
    ZIPの場合は安全に展開し、内部のディレクトリ構造に関わらず配下の.geojsonを再帰的に探索する
    （特定の配布元のファイル名・ディレクトリ名には依存しない）。サブフォルダがあればカテゴリとして
    hazard_typeに反映し、サブフォルダがなければ入力された基本種別名をそのままhazard_typeとする。
    利用者には集約したサマリのみ表示し、ファイルごとの詳細ログは出さない。"""
    suffix = Path(filename).suffix.lower()
    if suffix not in (".geojson", ".zip"):
        raise ValueError(f"'{filename}' は対応していない形式です。.geojson または .zip を選択してください。")

    if suffix == ".geojson":
        layers = []
        empty_file_count = 0
        layer = load_hazard_geojson(io.BytesIO(file_bytes), hazard_type)
        if len(layer) == 0:
            empty_file_count += 1
        else:
            layers.append(layer)
    else:
        with tempfile.TemporaryDirectory(prefix="hazard_zip_") as extract_dir_str:
            extract_dir = Path(extract_dir_str)
            safe_extract_zip(file_bytes, extract_dir)
            print(f"'{filename}' を展開しました。")

            geojson_paths = sorted(extract_dir.rglob("*.geojson"))
            if not geojson_paths:
                raise RuntimeError(f"'{filename}' 内に.geojsonファイルが見つかりませんでした。")
            print(f"GeoJSONを {len(geojson_paths)}ファイル検出しました。")

            categories = sorted(
                {
                    p.relative_to(extract_dir).parts[0]
                    for p in geojson_paths
                    if len(p.relative_to(extract_dir).parts) > 1
                }
            )
            if categories:
                print(f"{len(categories)}カテゴリ（サブフォルダ）を検出しました。")

            layers = []
            empty_file_count = 0
            for geojson_path in geojson_paths:
                file_hazard_type = hazard_type_for_path(hazard_type, geojson_path, extract_dir)
                layer = load_hazard_geojson(geojson_path, file_hazard_type)
                if len(layer) == 0:
                    empty_file_count += 1
                else:
                    layers.append(layer)

    if empty_file_count:
        print(f"{empty_file_count}ファイルは有効なPolygon/MultiPolygonを含まないため除外しました。")

    if not layers:
        raise RuntimeError(
            f"'{filename}' から有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
        )

    combined = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326")
    print(f"有効なハザードポリゴンを {len(combined)}件読み込みました。")
    combined_hazard_types = sorted(combined["hazard_type"].unique())
    if len(combined_hazard_types) == 1:
        print(f"hazard_type='{combined_hazard_types[0]}'")
    else:
        print(f"hazard_type: {', '.join(combined_hazard_types)}")
    return combined


hazard_gdf = None

if ENABLE_HAZARD_CHECK:
    print("ハザード区域のデータ（.geojson または国・県等の公式配布ZIP）をアップロードしてください（複数選択可）。")
    uploaded_hazards = files.upload()

    hazard_layers = []
    for hazard_filename, hazard_bytes in uploaded_hazards.items():
        hazard_type = input(
            f"'{hazard_filename}' の基本ハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮。"
            "ZIP内にサブフォルダがあれば、フォルダ名がカテゴリとして自動的に追加されます）: "
        ).strip()
        if not hazard_type:
            hazard_type = hazard_filename
        layer = load_hazard_upload(hazard_filename, hazard_bytes, hazard_type)
        hazard_layers.append(layer)

    if hazard_layers:
        hazard_gdf = gpd.GeoDataFrame(pd.concat(hazard_layers, ignore_index=True), crs="EPSG:4326")

    if hazard_gdf is None or len(hazard_gdf) == 0:
        raise RuntimeError(
            "ENABLE_HAZARD_CHECK=True ですが、有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
            "GeoJSON/ZIPファイルが正しくアップロードされているか、ジオメトリ形式を確認してください。"
            "ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください。"
        )

    print(f"ハザードデータを合計 {len(hazard_gdf)}件読み込みました。")
else:
    print("ENABLE_HAZARD_CHECK=False のため、ハザードデータの読込をスキップします。")


In [ ]:
# ===== 距離計算・ハザード判定関数 =====
# 各要支援者について、有効な避難所すべてとの直線距離（geopy.distance.geodesic、メートル単位）を計算し、
# 近い順に TOP_N 件を候補として算出します（道路距離ではありません）。並び替えは丸める前の距離で行い、
# メートル単位への丸め（小数1桁）はCSVに出力する値を作成する際にのみ行います。距離が同一の場合でも
# 結果順が実行ごとにばらつかないよう、避難所名を用いて順序を安定させます。
#
# ハザード判定関数は、要支援者地点・候補避難所地点がハザード区域の内部または境界上にあるか、また
# 両地点を結ぶ直線がハザード区域と交差するかを判定します（直線交差は道路上の避難経路判定ではありません）。
# 座標が欠損・範囲外で判定できない場合は、区域外(False)と混同しないよう NaN を返します。


def compute_candidates(resident_lat, resident_lon, shelters_df, top_n):
    """要支援者の座標から近い順に避難所候補を [(名前, 距離m, 緯度, 経度, 災害種別対応区分), ...] で返す。
    座標が欠損、または緯度・経度が有効範囲外であればNoneを返す（geodesic()に不正値を渡さない）。
    距離は丸めずに返す（並び替え後、出力時にのみ丸める）。対応区分は災害種別列が無ければNoneのまま。"""
    if not is_valid_coordinate(resident_lat, resident_lon):
        return None

    has_disaster_info = "_disaster_support" in shelters_df.columns
    resident_coord = (resident_lat, resident_lon)
    records = []
    for _, shelter in shelters_df.iterrows():
        shelter_coord = (shelter["latitude"], shelter["longitude"])
        distance_m = geodesic(resident_coord, shelter_coord).meters
        disaster_support = shelter["_disaster_support"] if has_disaster_info else np.nan
        records.append(
            (shelter["name"], distance_m, shelter["latitude"], shelter["longitude"], disaster_support)
        )

    records.sort(key=lambda record: (record[1], record[0]))
    return records[:top_n]


def hazard_types_at_point(lat, lon, hazard_area):
    """座標がハザード区域の内部または境界上にあるかどうかと、該当するhazard_type（;区切り）を返す。
    座標が欠損・範囲外の場合は判定不能としてnp.nanを返す（区域外=Falseと混同しない）。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if not is_valid_coordinate(lat, lon):
        return np.nan, np.nan

    point = Point(lon, lat)
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(point), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def hazard_types_on_line(lat1, lon1, lat2, lon2, hazard_area):
    """2地点を結ぶ直線がハザード区域と交差するかどうかと、該当するhazard_typeを返す（道路経路上の判定ではない）。
    いずれかの座標が欠損・範囲外の場合は判定不能としてnp.nanを返す。"""
    if hazard_area is None or len(hazard_area) == 0:
        return False, ""
    if not is_valid_coordinate(lat1, lon1) or not is_valid_coordinate(lat2, lon2):
        return np.nan, np.nan

    line = LineString([(lon1, lat1), (lon2, lat2)])
    hit_types = sorted(hazard_area.loc[hazard_area.geometry.intersects(line), "hazard_type"].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


print("距離計算・ハザード判定関数を定義しました。")


In [ ]:
# ===== 候補算出とハザード判定の実行 =====
# 要支援者ごとに、避難所候補・距離・（ENABLE_HAZARD_CHECK=True の場合のみ）ハザード判定結果を組み立てます。
# 距離による候補順位はハザード判定結果によって変更されません。座標がない・不正な要支援者の行も削除せず、
# 候補・距離を空欄のまま保持します（match_status 列で ok / no_coordinates(座標欄が空欄) /
# invalid_coordinates(値はあるが範囲外) の状態を確認できます）。
# ハザード関連列は、有効な座標で実際に判定できた場合のみ True/False とし、要支援者座標が欠損・不正な場合や
# 候補自体が存在しない場合は NaN（判定不能）のまま出力します（候補避難所地点は常に有効座標を持つため、
# 候補が存在すれば通常どおり True/False で判定します）。
# 避難所一覧に災害種別列があった場合は candidate_N_disaster_support 列に、災害種別ごとの対応区分
# （例: 洪水:対応済み;高潮:2階以上であれば対応済み）を出力します（対応区分によって候補の順位変更・自動除外は行いません）。

result_records = []

for _, resident in residents.iterrows():
    lat, lon = resident["latitude"], resident["longitude"]
    candidates = compute_candidates(lat, lon, shelters_valid, TOP_N)

    if pd.isna(lat) or pd.isna(lon):
        coord_status = "no_coordinates"
    elif not is_valid_coordinate(lat, lon):
        coord_status = "invalid_coordinates"
    else:
        coord_status = "ok"

    record = {}

    if ENABLE_HAZARD_CHECK:
        in_hazard, hazard_types = hazard_types_at_point(lat, lon, hazard_gdf)
        record["resident_in_hazard"] = in_hazard
        record["resident_hazard_types"] = hazard_types

    for i in range(TOP_N):
        n = i + 1
        candidate_col = f"candidate_{n}"
        distance_col = f"distance_{n}_m"
        disaster_col = f"candidate_{n}_disaster_support"
        shelter_hazard_col = f"candidate_{n}_shelter_in_hazard"
        shelter_hazard_types_col = f"candidate_{n}_shelter_hazard_types"
        line_hazard_col = f"candidate_{n}_straight_line_intersects_hazard"
        line_hazard_types_col = f"candidate_{n}_straight_line_hazard_types"

        if candidates is not None and i < len(candidates):
            shelter_name, distance_m, shelter_lat, shelter_lon, disaster_support = candidates[i]
            record[candidate_col] = shelter_name
            record[distance_col] = round(distance_m, 1)

            if HAS_DISASTER_TYPE_COLUMNS:
                record[disaster_col] = disaster_support

            if ENABLE_HAZARD_CHECK:
                shelter_in_hazard, shelter_hazard_types = hazard_types_at_point(
                    shelter_lat, shelter_lon, hazard_gdf
                )
                record[shelter_hazard_col] = shelter_in_hazard
                record[shelter_hazard_types_col] = shelter_hazard_types

                line_intersects, line_hazard_types = hazard_types_on_line(
                    lat, lon, shelter_lat, shelter_lon, hazard_gdf
                )
                record[line_hazard_col] = line_intersects
                record[line_hazard_types_col] = line_hazard_types
        else:
            record[candidate_col] = np.nan
            record[distance_col] = np.nan
            if HAS_DISASTER_TYPE_COLUMNS:
                record[disaster_col] = np.nan
            if ENABLE_HAZARD_CHECK:
                record[shelter_hazard_col] = np.nan
                record[shelter_hazard_types_col] = np.nan
                record[line_hazard_col] = np.nan
                record[line_hazard_types_col] = np.nan

    record["match_status"] = coord_status
    result_records.append(record)

results_df = pd.DataFrame(result_records)
final_df = pd.concat([residents.reset_index(drop=True), results_df], axis=1)

print("候補算出が完了しました。")


In [ ]:
# ===== 結果確認・CSV出力 =====
# CSVを出力する前に、Notebook上で処理結果の概要と先頭数行を確認します。
# 結果は Excelで文字化けしにくい utf-8-sig（UTF-8 BOM付き）でCSVに出力し、ブラウザへダウンロードします。
# 出力CSVには個人情報が含まれ得るため、取り扱いに注意してください。

total_residents = len(final_df)
ok_count = int((final_df["match_status"] == "ok").sum())
no_coord_count = int((final_df["match_status"] == "no_coordinates").sum())
invalid_coord_count = int((final_df["match_status"] == "invalid_coordinates").sum())

print(f"要支援者件数: {total_residents}件")
print(f"距離計算できた件数: {ok_count}件")
print(f"座標が空欄のため距離計算できなかった件数: {no_coord_count}件")
print(f"座標が範囲外で不正なため距離計算できなかった件数: {invalid_coord_count}件")
print(f"距離計算に使用した有効な避難所件数: {len(shelters_valid)}件")

if ENABLE_HAZARD_CHECK:
    resident_hazard_count = int((final_df["resident_in_hazard"] == True).sum())
    print(f"ハザード区域内（境界上含む）にいる要支援者数: {resident_hazard_count}件")

    shelter_hazard_flags = [
        final_df[f"candidate_{i + 1}_shelter_in_hazard"] == True for i in range(TOP_N)
    ]
    line_hazard_flags = [
        final_df[f"candidate_{i + 1}_straight_line_intersects_hazard"] == True for i in range(TOP_N)
    ]
    shelter_hazard_count = int(pd.concat(shelter_hazard_flags, axis=1).sum().sum())
    line_hazard_count = int(pd.concat(line_hazard_flags, axis=1).sum().sum())

    print(f"ハザード区域内にある候補避難所の件数（延べ、TOP_N分の合計）: {shelter_hazard_count}件")
    print(f"候補避難所への直線がハザード区域と交差する件数（延べ、TOP_N分の合計）: {line_hazard_count}件")

display(final_df.head())

OUTPUT_FILENAME = "assigned_shelters.csv"

final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
print(f"'{OUTPUT_FILENAME}' を出力しました。")

files.download(OUTPUT_FILENAME)
